# ATLAS wind preprocessing

This notebook prepares wind reanalysis data for the ATLAS workflow.

The goal is to combine ERA5 Land and ERA5 on the same grid, using ERA5 to fill missing ERA5 Land values, especially along coastal areas.

The notebook is designed to be run step by step by users with limited Python experience. The only section that should normally be edited is **User configuration**.

## 1. What this notebook does

The workflow is: 

1. read ERA5 Land and ERA5 NetCDF files for the selected wind component
2. check and standardize coordinates and longitudes
3. compute daily means from hourly data
4. save two intermediate daily files
5. interpolate ERA5 onto the ERA5 Land grid
6. use ERA5 to fill missing values in ERA5 Land
7. clip the dataset to the study area
8. save the final preprocessed file

## 2. Import libraries

Run this cell without editing it.

If a library is missing, install it in your Python environment before continuing.

In [1]:
from pathlib import Path
from datetime import datetime
import glob

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

try:
    import rioxarray  # noqa: F401
    RIOXARRAY_AVAILABLE = True
except ImportError:
    RIOXARRAY_AVAILABLE = False
    print("Warning: rioxarray is not installed. The notebook may still work if the data already have correct longitude/latitude coordinates.")


## 3. User configuration

Edit only this cell.

The paths are generic. You can use relative paths, for example `../data/...`, or absolute paths, for example `/home/user/project/data/...`.

For the wind component use:

`u` for the zonal component

`v` for the meridional component

In [2]:
# Country or study area
COUNTRY = "chile"

# Wind component: "u" or "v"
COMPONENT = "v"

# Variable name in the NetCDF files.
# For ERA5 and ERA5 Land this is usually "u10" or "v10".
VARIABLE_NAME = f"{COMPONENT}10"

# Period label used only to build output file names
PERIOD_LABEL = "1996-01_2025-12"

# Main project folder
PROJECT_DIR = Path("../")

# Folder containing the downloaded data
# Expected structure:
# DATA_DIR / "era5land" / COUNTRY / NetCDF files
# DATA_DIR / "era5" / COUNTRY / NetCDF files
DATA_DIR = PROJECT_DIR / "data" / "cds_downloads"

# Specific input folders
ERA5LAND_DIR = DATA_DIR / "era5land" / f"10m_{COMPONENT}_component_of_wind" / COUNTRY
ERA5_DIR = DATA_DIR / "era5" / f"10m_{COMPONENT}_component_of_wind" / COUNTRY

# Output folder
OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / f"10m_wind_{COMPONENT}_component" / COUNTRY

# Optional shapefile used to clip the dataset.
# Set this to None if you want to use the manual bounding box instead.
SHAPEFILE_PATH = PROJECT_DIR / "data" / "shapefiles" / COUNTRY / "REGIONES_v1.shp"

# Name of the shapefile column used to select the study area.
# If the shapefile already contains only the study area, keep None.
SHAPEFILE_NAME_COLUMN = None
SHAPEFILE_NAME_VALUE = None

# Manual bounding box used if the shapefile does not exist.
# Format: [lon_min, lat_min, lon_max, lat_max]
# Approximate example for Chile: [-76, -56, -66, -17]
MANUAL_BBOX = [-76, -56, -66, -17]

# Buffer around the study area, in degrees
BUFFER_DEG = 0.25

# If True, existing files will be overwritten
OVERWRITE = True

## 4. Check the configuration

This cell creates the output folder and prints the main paths.

If any input folder does not exist, correct the configuration in the previous cell.

In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Selected configuration")
print(f"Country: {COUNTRY}")
print(f"Component: {COMPONENT}")
print(f"Variable: {VARIABLE_NAME}")
print(f"ERA5 Land input: {ERA5LAND_DIR}")
print(f"ERA5 input: {ERA5_DIR}")
print(f"Output: {OUTPUT_DIR}")

for folder in [ERA5LAND_DIR, ERA5_DIR]:
    if not folder.exists():
        print(f"Warning: this folder does not exist: {folder}")

Selected configuration
Country: chile
Component: v
Variable: v10
ERA5 Land input: ../data/cds_downloads/era5land/10m_v_component_of_wind/chile
ERA5 input: ../data/cds_downloads/era5/10m_v_component_of_wind/chile
Output: ../data/processed/10m_wind_v_component/chile


## 5. Helper functions

Run this cell without editing it.

In [4]:
def roll_longitudes_to_minus180_180(ds, lon_name="longitude"):
    """Convert longitudes from 0/360 to -180/180, if needed."""
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360) - 180})
    return ds.sortby(lon_name)


def ensure_standard_coordinates(xdf):
    """Standardize the names, order and format of the main coordinates."""
    rename_map = {}
    if "lon" in xdf.coords and "longitude" not in xdf.coords:
        rename_map["lon"] = "longitude"
    if "lat" in xdf.coords and "latitude" not in xdf.coords:
        rename_map["lat"] = "latitude"
    if "valid_time" in xdf.coords and "time" not in xdf.coords:
        rename_map["valid_time"] = "time"
    if rename_map:
        xdf = xdf.rename(rename_map)

    required = {"longitude", "latitude", "time"}
    missing = required.difference(set(xdf.coords))
    if missing:
        raise ValueError(f"Missing coordinates in the dataset: {missing}")

    xdf = xdf.sortby("latitude")
    xdf = xdf.sortby("longitude")

    if float(xdf.longitude.min()) >= 0:
        xdf = roll_longitudes_to_minus180_180(xdf)

    xdf = xdf.assign_coords(
        longitude=np.round(xdf.longitude.astype(float), 3),
        latitude=np.round(xdf.latitude.astype(float), 3),
    )

    if RIOXARRAY_AVAILABLE:
        try:
            xdf = xdf.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
            if xdf.rio.crs is None:
                xdf = xdf.rio.write_crs("EPSG:4326", inplace=False)
        except Exception as exc:
            print(f"Note: CRS was not assigned with rioxarray. Details: {exc}")

    return xdf


def list_netcdf_files(folder, component):
    """Search for NetCDF files matching the selected component."""
    folder = Path(folder)
    patterns = [f"*_{component}*.nc", f"*{component}10*.nc", "*.nc"]
    files = []
    for pattern in patterns:
        files = sorted(folder.glob(pattern))
        if files:
            break
    return files


def load_data(folder, component):
    """Load all NetCDF files found in the selected folder."""
    files = list_netcdf_files(folder, component)
    if not files:
        raise FileNotFoundError(f"No NetCDF files found in: {folder}")

    print(f"Found {len(files)} files in {folder}")
    for file in files[:5]:
        print(f"{file.name}")
    if len(files) > 5:
        print("  ...")

    return xr.open_mfdataset(
        [str(file) for file in files],
        engine="netcdf4",
        combine="by_coords",
        preprocess=ensure_standard_coordinates,
    )


def aggregate_daymean(xdf, variable_name):
    """Compute the daily mean of the selected variable."""
    xdf = ensure_standard_coordinates(xdf)

    if variable_name not in xdf.data_vars:
        available = list(xdf.data_vars)
        raise ValueError(f"Variable {variable_name} not found. Available variables: {available}")

    return xdf[variable_name].resample(time="1D").mean()


def load_geometries(shapefile_path=None, bbox=None, name_column=None, name_value=None):
    """Load geometries from a shapefile or create a geometry from a bounding box."""
    if shapefile_path is not None and Path(shapefile_path).exists():
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
        else:
            gdf = gdf.to_crs(epsg=4326)

        if name_column is not None and name_value is not None:
            gdf = gdf[gdf[name_column] == name_value]
            if gdf.empty:
                raise ValueError(f"No geometry found with {name_column} = {name_value}")

        return gdf.geometry

    if bbox is None:
        raise ValueError("A valid shapefile or a manual bounding box is required.")

    lon_min, lat_min, lon_max, lat_max = bbox
    return gpd.GeoSeries.from_bbox((lon_min, lat_min, lon_max, lat_max), crs="EPSG:4326")


def cut_xdf(xdf, geometries, buffer_deg=0.0, buffer_cells=None):
    """Clip the dataset using the bounding box of the geometries."""
    xdf = ensure_standard_coordinates(xdf)

    if hasattr(geometries, "total_bounds"):
        lon_min, lat_min, lon_max, lat_max = geometries.total_bounds
    else:
        lon_min, lat_min, lon_max, lat_max = geometries.bounds

    if buffer_cells is not None:
        if xdf.longitude.size < 2 or xdf.latitude.size < 2:
            raise ValueError("Unable to compute the grid resolution.")
        dlon = float(np.abs(xdf.longitude.values[1] - xdf.longitude.values[0]))
        dlat = float(np.abs(xdf.latitude.values[1] - xdf.latitude.values[0]))
        lon_buffer = buffer_cells * dlon
        lat_buffer = buffer_cells * dlat
    else:
        lon_buffer = buffer_deg
        lat_buffer = buffer_deg

    return xdf.sel(
        longitude=slice(lon_min - lon_buffer, lon_max + lon_buffer),
        latitude=slice(lat_min - lat_buffer, lat_max + lat_buffer),
    )


def save_netcdf(xdf, output_filename, variable_name=None, overwrite=True):
    """Save a Dataset or DataArray to NetCDF using float32 dtype."""
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)

    if output_filename.exists() and not overwrite:
        print(f"File already exists and was not overwritten: {output_filename}")
        return

    for coord in ["number", "expver"]:
        if coord in xdf.coords:
            xdf = xdf.drop_vars(coord)

    if isinstance(xdf, xr.DataArray):
        if xdf.name is None:
            xdf = xdf.rename(variable_name or "variable")
        encoding = {xdf.name: {"dtype": "float32"}}
    else:
        encoding = {var: {"dtype": "float32"} for var in xdf.data_vars}

    chunk_dict = {dim: size for dim, size in {"time": 50, "latitude": 256, "longitude": 256}.items() if dim in xdf.dims}
    if chunk_dict:
        xdf = xdf.chunk(chunk_dict)

    xdf.to_netcdf(output_filename, engine="netcdf4", encoding=encoding)
    print(f"File saved: {output_filename}")


## 6. Load the study area

This cell loads the shapefile if it exists. Otherwise, it uses the manual bounding box defined in the configuration.

In [5]:
geometries = load_geometries(
    shapefile_path=SHAPEFILE_PATH,
    bbox=MANUAL_BBOX,
    name_column=SHAPEFILE_NAME_COLUMN,
    name_value=SHAPEFILE_NAME_VALUE,
)

print("Study area loaded")
print(f"Bounding box: {geometries.total_bounds}")


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


Study area loaded
Bounding box: [-109.45491616  -56.53776582  -66.41559401  -17.49839934]


## 7. Load ERA5 Land and ERA5 data

This cell may take some time if there are many files or if the files are large.

In [6]:
era5land_ds = load_data(ERA5LAND_DIR, COMPONENT)
era5_ds = load_data(ERA5_DIR, COMPONENT)

print("ERA5 Land")
print(era5land_ds)
print("ERA5")
print(era5_ds)


Found 1 files in ../data/cds_downloads/era5land/10m_v_component_of_wind/chile
era5land_10m_v_component_of_wind_1991_01.nc
Found 1 files in ../data/cds_downloads/era5/10m_v_component_of_wind/chile
era5_10m_v_component_of_wind_1991_01.nc
ERA5 Land
<xarray.Dataset> Size: 576MB
Dimensions:      (time: 744, latitude: 411, longitude: 471)
Coordinates:
    number       int64 8B 0
  * time         (time) datetime64[ns] 6kB 1991-01-01 ... 1991-01-31T23:00:00
    expver       (time) <U4 12kB dask.array<chunksize=(744,), meta=np.ndarray>
  * longitude    (longitude) float64 4kB -112.0 -111.9 -111.8 ... -65.1 -65.0
  * latitude     (latitude) float64 3kB -57.0 -56.9 -56.8 ... -16.2 -16.1 -16.0
    spatial_ref  int64 8B 0
Data variables:
    v10          (time, latitude, longitude) float32 576MB dask.array<chunksize=(186, 102, 118), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:      

## 8. Daily aggregation

Hourly data are converted into daily means.

Intermediate files are saved because they make the final step faster if the notebook needs to be run again.

In [7]:
era5land_daily = aggregate_daymean(era5land_ds, VARIABLE_NAME)
era5_daily = aggregate_daymean(era5_ds, VARIABLE_NAME)

print("Daily aggregation completed")
print(era5land_daily)
print(era5_daily)

era5land_step1_file = OUTPUT_DIR / f"era5land_{VARIABLE_NAME}_{PERIOD_LABEL}_step1.nc"
era5_step1_file = OUTPUT_DIR / f"era5_{VARIABLE_NAME}_{PERIOD_LABEL}_step1.nc"

save_netcdf(era5land_daily, era5land_step1_file, variable_name=VARIABLE_NAME, overwrite=OVERWRITE)
save_netcdf(era5_daily, era5_step1_file, variable_name=VARIABLE_NAME, overwrite=OVERWRITE)


Daily aggregation completed
<xarray.DataArray 'v10' (time: 31, latitude: 411, longitude: 471)> Size: 24MB
dask.array<stack, shape=(31, 411, 471), dtype=float32, chunksize=(1, 103, 118), chunktype=numpy.ndarray>
Coordinates:
    number       int64 8B 0
    spatial_ref  int64 8B 0
  * longitude    (longitude) float64 4kB -112.0 -111.9 -111.8 ... -65.1 -65.0
  * latitude     (latitude) float64 3kB -57.0 -56.9 -56.8 ... -16.2 -16.1 -16.0
  * time         (time) datetime64[ns] 248B 1991-01-01 1991-01-02 ... 1991-01-31
Attributes: (12/32)
    GRIB_paramId:                             166
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      193581
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
 

## 9. Interpolation and merge

ERA5 is interpolated onto the ERA5 Land grid.

Then ERA5 Land is kept where available, while ERA5 is used to fill missing values.

In [8]:
era5land_step1 = xr.open_dataset(era5land_step1_file)
era5_step1 = xr.open_dataset(era5_step1_file)

era5_interp = era5_step1.interp(
    latitude=era5land_step1.latitude.values,
    longitude=era5land_step1.longitude.values,
    method="slinear",
)

merged = era5land_step1.combine_first(era5_interp)
merged_cut = cut_xdf(merged, geometries, buffer_deg=BUFFER_DEG)

print("Merge completed")
print(merged_cut)


Merge completed
<xarray.Dataset> Size: 43MB
Dimensions:      (time: 31, latitude: 395, longitude: 436)
Coordinates:
    spatial_ref  int64 8B 0
  * time         (time) datetime64[ns] 248B 1991-01-01 1991-01-02 ... 1991-01-31
  * longitude    (longitude) float64 3kB -109.7 -109.6 -109.5 ... -66.3 -66.2
  * latitude     (latitude) float64 3kB -56.7 -56.6 -56.5 ... -17.5 -17.4 -17.3
Data variables:
    v10          (time, latitude, longitude) float64 43MB -0.9571 ... -0.14


## 10. Save the final file

The final file contains the daily preprocessed dataset, clipped to the study area.

In [9]:
final_file = OUTPUT_DIR / f"{VARIABLE_NAME}_{PERIOD_LABEL}_processed.nc"
save_netcdf(merged_cut, final_file, variable_name=VARIABLE_NAME, overwrite=OVERWRITE)

print("Preprocessing completed")
print(f"Final file: {final_file}")


File saved: ../data/processed/10m_wind_v_component/chile/v10_1991-01_2020-12_processed.nc
Preprocessing completed
Final file: ../data/processed/10m_wind_v_component/chile/v10_1991-01_2020-12_processed.nc


## 11. Quick output check

This cell reopens the final file and prints useful information to check that the preprocessing completed successfully.

In [10]:
check_ds = xr.open_dataset(final_file)
print(check_ds)

if VARIABLE_NAME in check_ds:
    print("Minimum and maximum values")
    print(float(check_ds[VARIABLE_NAME].min()), float(check_ds[VARIABLE_NAME].max()))
else:
    print(f"Warning: variable {VARIABLE_NAME} is not present in the final file.")


<xarray.Dataset> Size: 21MB
Dimensions:      (time: 31, latitude: 395, longitude: 436)
Coordinates:
    spatial_ref  int64 8B ...
  * time         (time) datetime64[ns] 248B 1991-01-01 1991-01-02 ... 1991-01-31
  * longitude    (longitude) float64 3kB -109.7 -109.6 -109.5 ... -66.3 -66.2
  * latitude     (latitude) float64 3kB -56.7 -56.6 -56.5 ... -17.5 -17.4 -17.3
Data variables:
    v10          (time, latitude, longitude) float32 21MB ...
Minimum and maximum values
-16.774751663208008 14.957385063171387
